# [Improved V1] NFL Draft Prediction

Improvements over baseline:
- **LightGBM** instead of RandomForest
- **Feature Engineering**: BMI, missing-value flags, position-relative z-scores
- **StratifiedKFold** cross-validation with early stopping

## 1. Setup

In [10]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier 
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import lightgbm as lgb

## 2. Load the Data

In [11]:
train = pd.read_csv('input/train.csv')
test  = pd.read_csv('input/test.csv')

print('Train shape:', train.shape)
print('Test shape: ', test.shape)
train.head()

Train shape: (2781, 16)
Test shape:  (696, 15)


,Id,Year,Age,School,Height,Weight,Sprint_40yd,Vertical_Jump,Bench_Press_Reps,Broad_Jump,Agility_3cone,Shuttle,Player_Type,Position_Type,Position,Drafted
0,0,2011,21.0,Lehigh,1.9050,140.160042,5.39,59.69,29.0,251.46,7.91,4.94,offense,offensive_lineman,OG,1.0
1,1,2011,24.0,Abilene Christian,1.8288,87.089735,4.31,101.60,16.0,332.74,NaN,NaN,offense,backs_receivers,WR,1.0
2,2,2018,21.0,Colorado St.,1.8542,92.986436,4.51,91.44,10.0,309.88,6.95,4.37,offense,backs_receivers,WR,1.0
3,3,2010,21.0,East Carolina,1.9304,148.778297,5.09,76.20,39.0,254.00,8.12,4.71,defense,defensive_lineman,DT,1.0
4,4,2016,21.0,California,1.8796,92.079251,4.64,78.74,NaN,281.94,7.13,4.20,offense,backs_receivers,WR,1.0


## 3. Data Analysis & EDA

In [12]:
# Check missing values
print('Missing values in train:')
print(train.isnull().sum())
print()
print('Target distribution:')
print(train['Drafted'].value_counts())

Missing values in train:
Id                    0
Year                  0
Age                 435
School                0
Height                0
Weight                0
Sprint_40yd         145
Vertical_Jump       554
Bench_Press_Reps    721
Broad_Jump          581
Agility_3cone       970
Shuttle             912
Player_Type           0
Position_Type         0
Position              0
Drafted               0
dtype: int64

Target distribution:
Drafted
1.0    1803
0.0     978
Name: count, dtype: int64


In [13]:
# Performance columns
perf_cols = ['Sprint_40yd', 'Vertical_Jump', 'Bench_Press_Reps',
             'Broad_Jump', 'Agility_3cone', 'Shuttle']

# Missing rate per column
print('Missing rate per performance column (train):')
for col in perf_cols:
    rate = train[col].isnull().mean()
    print(f'  {col}: {rate:.1%}')

Missing rate per performance column (train):
  Sprint_40yd: 5.2%
  Vertical_Jump: 19.9%
  Bench_Press_Reps: 25.9%
  Broad_Jump: 20.9%
  Agility_3cone: 34.9%
  Shuttle: 32.8%


## 4. Preprocessing & Feature Engineering

In [14]:
def engineer_features(df):
    df = df.copy()

    # BMI
    df['BMI'] = df['Weight'] / (df['Height'] ** 2)

    # Missing-value flags
    perf_cols = ['Sprint_40yd', 'Vertical_Jump', 'Bench_Press_Reps',
                 'Broad_Jump', 'Agility_3cone', 'Shuttle']
    for col in perf_cols:
        df[f'{col}_missing'] = df[col].isna().astype(int)

    # Number of missing performance tests
    df['n_missing'] = df[perf_cols].isna().sum(axis=1)

    # Position-relative z-scores
    for col in perf_cols:
        grp = df.groupby('Position')[col]
        df[f'{col}_pos_zscore'] = (df[col] - grp.transform('mean')) / (grp.transform('std') + 1e-6)

    return df

train = engineer_features(train)
test  = engineer_features(test)

print('New features added. Train shape:', train.shape)

New features added. Train shape: (2781, 30)


In [15]:
# Encode categoricals
cat_cols = ['Player_Type', 'Position_Type', 'Position']
for col in cat_cols:
    le = LabelEncoder()
    combined = pd.concat([train[col], test[col]], axis=0).astype(str)
    le.fit(combined)
    train[col] = le.transform(train[col].astype(str))
    test[col]  = le.transform(test[col].astype(str))

print('Categorical encoding done.')

Categorical encoding done.


In [16]:
# Define features
base_num = ['Year', 'Age', 'Height', 'Weight',
            'Sprint_40yd', 'Vertical_Jump', 'Bench_Press_Reps',
            'Broad_Jump', 'Agility_3cone', 'Shuttle']

engineered = ['BMI', 'n_missing'] + \
             [f'{c}_missing'    for c in ['Sprint_40yd','Vertical_Jump','Bench_Press_Reps',
                                          'Broad_Jump','Agility_3cone','Shuttle']] + \
             [f'{c}_pos_zscore' for c in ['Sprint_40yd','Vertical_Jump','Bench_Press_Reps',
                                          'Broad_Jump','Agility_3cone','Shuttle']]

features = base_num + cat_cols + engineered

X      = train[features]
y      = train['Drafted']
X_test = test[features]

print(f'Number of features: {len(features)}')

Number of features: 27


## 5. Baseline Model (LightGBM)

In [17]:
params = {
    'objective':         'binary',
    'metric':            'auc',
    'learning_rate':     0.05,
    'num_leaves':        63,
    'min_child_samples': 20,
    'feature_fraction':  0.8,
    'bagging_fraction':  0.8,
    'bagging_freq':      5,
    'verbose':           -1,
    'n_jobs':            -1,
}

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_preds  = np.zeros(len(X))
test_preds = np.zeros(len(X_test))

for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

    dtrain = lgb.Dataset(X_tr, label=y_tr)
    dval   = lgb.Dataset(X_val, label=y_val, reference=dtrain)

    model = lgb.train(
        params,
        dtrain,
        num_boost_round=1000,
        valid_sets=[dval],
        callbacks=[lgb.early_stopping(50, verbose=False),
                   lgb.log_evaluation(period=-1)],
    )

    oof_preds[val_idx] = model.predict(X_val)
    test_preds        += model.predict(X_test) / 5

    fold_auc = roc_auc_score(y_val, oof_preds[val_idx])
    print(f'Fold {fold+1}  AUC: {fold_auc:.5f}')

oof_auc = roc_auc_score(y, oof_preds)
print(f'\nOverall OOF AUC: {oof_auc:.5f}')

Fold 1  AUC: 0.79419
Fold 2  AUC: 0.84793
Fold 3  AUC: 0.86722
Fold 4  AUC: 0.79041
Fold 5  AUC: 0.83241

Overall OOF AUC: 0.81492


## 6. Hypothesis & Feature Engineering

Key ideas used:
- **BMI**: body mass index from height and weight
- **Missing flags**: whether a player skipped a drill (itself informative)
- **n_missing**: total number of skipped drills
- **Position z-scores**: how good is this player *relative to others at the same position*

## 7. Create the Submission File

In [18]:
submission = pd.read_csv('input/sample_submission.csv')
submission['Drafted'] = test_preds
submission.to_csv('submission_V1.csv', index=False)
print('submission_V1.csv saved!')
submission.head()

submission_V1.csv saved!


,Id,Drafted
0,2781,0.649263
1,2782,0.827671
2,2783,0.775348
3,2784,0.830935
4,2785,0.737416


## 8. Next Steps

- **V2**: Hyperparameter tuning (Optuna or grid search)
- **V3**: More feature engineering (cross-features, school encoding, etc.)
- **V4**: Ensemble of multiple models